# Modular Inference Pipeline

Single-path pipeline for `Qwen/Qwen3-4B-Thinking-2507` on the CSE 151B math reasoning competition.

Design goals (vs the starter / `system_prompts.ipynb`):

1. **MCQ that always produces a letter.** Short token cap (`MAX_TOKENS_MCQ = 256`), `/no_think` suffix, greedy decoding with a repetition penalty, an early stop on the first `FINAL: <letter>`, and a deterministic single-letter fallback retry if the primary pass still produces nothing parseable.
2. **Free-form that does not spiral.** Thinking is allowed but capped: `BudgetForcingProcessor` force-injects `</think>` after `THINK_BUDGET_FREE = 4096` thinking tokens, and a new `BoxedStopCriteria` halts generation as soon as a complete `\boxed{...}` is emitted after `</think>`.
3. **One linear path, no toggles.** A single `solve_item(item)` entry point routes MCQ vs free-form internally. No `RUN_METHOD_SWEEP`, no commented-out alternative loaders, no per-method branches in the main run.

Inputs:
- `../data/public.jsonl` — questions with ground-truth answers.

Outputs:
- `results/modular_results.jsonl` — one record per question with `id`, `is_mcq`, `gold`, `response`, `correct`.

## 1. Configuration

All knobs live here. Behaviour is fully determined by these constants plus the public dataset.

In [1]:
!source ../.venv/bin/activate

In [10]:
import json
import os
import re
import sys
import time
from pathlib import Path
from typing import Optional

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"
DATA_PATH   = "../data/public.jsonl"
OUTPUT_PATH = "results/modular_results.jsonl"

# Set to None to score the full public set; set to an int to test on a slice.
LIMIT: Optional[int] = 5

# ── MCQ: short, decisive, /no_think ─────────────────────────────────────────
MAX_TOKENS_MCQ      = 256        # answer is a single line `FINAL: X`; 256 is plenty
TEMP_MCQ            = 0.0        # greedy = most decisive, no spiral randomness
TOP_P_MCQ           = 1.0
TOP_K_MCQ           = 0
REP_PEN_MCQ         = 1.15       # suppresses `Wait`/`Hmm` repetition loops
MCQ_NO_THINK        = True       # appends `/no_think` to the user message
THINK_BUDGET_MCQ    = 64         # belt-and-suspenders if /no_think is ignored
FALLBACK_MAX_TOKENS = 16         # for the deterministic single-letter retry

# ── Free-form: thinking allowed but bounded ─────────────────────────────────
MAX_TOKENS_FREE   = 8192
TEMP_FREE         = 0.4          # lower than the Qwen3 default (0.6) to reduce drift
TOP_P_FREE        = 0.95
TOP_K_FREE        = 20
REP_PEN_FREE      = 1.05
THINK_BUDGET_FREE = 4096         # peak Level-5 accuracy plateau per Muennighoff et al.

# Whether the saved JSONL should include `gold` and `correct` (True for public, False for private).
SAVE_EVAL = True

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    LogitsProcessor,
    StoppingCriteria,
    StoppingCriteriaList,
)
from tqdm import tqdm

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device    : {torch.cuda.get_device_name(0)}")

CUDA available : True
CUDA device    : NVIDIA A30


## 2. Load the public dataset

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]
n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 1126 questions  (375 MCQ, 751 free-form)


## 3. Load the model

Transformers + 4-bit BitsAndBytes quantization (single backend; vLLM is not used in this notebook).
After loading we resolve the `</think>` token id once for use by the budget-forcing logits processor.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

_eot_ids = tokenizer.encode("</think>", add_special_tokens=False)
END_THINK_TOKEN_ID = _eot_ids[0] if len(_eot_ids) == 1 else tokenizer.convert_tokens_to_ids("</think>")
print(f"</think> token id: {END_THINK_TOKEN_ID}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

</think> token id: 151668


## 4. Prompts

Two system prompts encode the desired behaviour:

- **MCQ**: timed-exam framing, explicit ban on hedging filler, instruction to pick the closest match if the computed value isn't an exact option, strict output format `FINAL: X`.
- **Free-form**: one-pass instruction (no re-verification of confident steps), single `\boxed{}` final answer, comma-separated sub-answers inside one box.

The MCQ user prompt is wrapped with a one-shot example that anchors the exact format and a trailing format reminder (models attend most strongly to the start and end of the user turn). When `MCQ_NO_THINK` is true the user message ends with `/no_think`, which Qwen3 interprets as a request to skip the `<think>` block.

In [5]:
SYSTEM_PROMPT_MCQ = (
    "You are solving a multiple-choice math exam under time pressure. "
    "Assess difficulty first: if the problem is standard, solve in one short path. "
    "Do NOT spiral: do not re-derive, re-check, or second-guess a clean computation. "
    "Do NOT use hedging filler (e.g. 'Wait', 'Hmm', 'Actually', 'Let me reconsider'). "
    "If your computed value does not exactly match an option, pick the closest match. "
    "Return your final output as exactly one line in this format: FINAL: X "
    "where X is a single option letter from A to J. "
    "No explanation, no extra words, no punctuation after the letter."
)

SYSTEM_PROMPT_FREE = (
    "You are an expert mathematician solving a timed exam. "
    "Assess difficulty first: trivial problems need only a few lines, not a derivation. "
    "Solve step by step but do NOT re-verify steps you are already confident about. "
    "Do each calculation once. State results directly when arithmetic is straightforward. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside one \\boxed{}, "
    "e.g. \\boxed{3, 7}. "
    "Stop generating immediately once \\boxed{} contains your final answer."
)

MCQ_FEWSHOT = (
    "Example:\n"
    "Q: What is 2+3?\n"
    "A. 4\nB. 5\nC. 6\nD. 7\n\n"
    "FINAL: B\n\n"
    "Now solve the following problem.\n\n"
)


def build_mcq_user(question: str, options: list, no_think: bool) -> str:
    """Format an MCQ as a user-turn message with anchor example and strict format reminder."""
    labels    = [chr(65 + i) for i in range(len(options))]
    opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
    suffix    = "\n/no_think" if no_think else ""
    return (
        f"{MCQ_FEWSHOT}"
        f"{question}\n\n"
        f"Options:\n{opts_text}\n\n"
        "Reasoning style (internal): at most a few short steps; no hedging words; "
        "no repeating the same check; commit to one letter.\n\n"
        "Output format (strict): FINAL: X\n"
        "X must be one letter from A-J.\n"
        "Do not output any other text."
        f"{suffix}"
    )


def build_free_user(question: str) -> str:
    """Free-form questions: pass the question through; thinking remains enabled."""
    return question


# Quick sanity check on one of each type.
_mcq_sample  = next(d for d in data if d.get("options"))
_free_sample = next(d for d in data if not d.get("options"))
print("── MCQ user prompt (first 300 chars) ──")
print(build_mcq_user(_mcq_sample["question"], _mcq_sample["options"], no_think=MCQ_NO_THINK)[:300], "...")
print("\n── Free-form user prompt ──")
print(build_free_user(_free_sample["question"]))

── MCQ user prompt (first 300 chars) ──
Example:
Q: What is 2+3?
A. 4
B. 5
C. 6
D. 7

FINAL: B

Now solve the following problem.

$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. $frac{1}{ ...

── Free-form user prompt ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]


## 5. Generation primitives

Three guards plus four extractors:

- `BudgetForcingProcessor`: when the number of generated tokens reaches `think_budget` and `</think>` has not yet appeared in the output, masks all logits to `-inf` except `</think>`, forcing the model to close its thinking block. Implements Muennighoff et al. (2025) budget forcing.
- `FinalAnswerStopCriteria`: stops generation the moment the visible output contains `FINAL: <letter>` or `\boxed{<letter>}`. Used for MCQs.
- `BoxedStopCriteria`: stops generation the moment a complete `\boxed{...}` (with balanced braces) appears **after** `</think>`. Used for free-form. Correctly handles nested LaTeX braces such as `\boxed{\frac{1}{2}}`.
- `strip_thinking`: extracts the visible answer. Robust against the case where the budget processor injects a stray `</think>` on top of an already-visible answer (it then takes the text **before** the `</think>`).
- `extract_mcq_letter`, `extract_boxed`, `clean_visible`: parsing helpers used by the solver and scorer.

In [6]:
class BudgetForcingProcessor(LogitsProcessor):
    """Force-inject `</think>` once the thinking budget is exceeded."""

    def __init__(self, end_think_token_id: int, think_budget: int, prompt_length: int):
        self.end_think_token_id = end_think_token_id
        self.think_budget = think_budget
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores):
        n_generated = input_ids.shape[1] - self.prompt_length
        if self.end_think_token_id in input_ids[0, self.prompt_length:].tolist():
            return scores
        if n_generated >= self.think_budget:
            scores[:, :] = float("-inf")
            scores[:, self.end_think_token_id] = 0.0
        return scores


class FinalAnswerStopCriteria(StoppingCriteria):
    """Stop MCQ generation as soon as `FINAL: <letter>` or `\\boxed{<letter>}` appears."""

    _PATTERNS = [
        re.compile(r"FINAL\s*:\s*([A-J])\b", flags=re.IGNORECASE),
        re.compile(r"\\boxed\{\s*([A-J])\s*\}", flags=re.IGNORECASE),
    ]

    def __init__(self, tokenizer, prompt_length: int):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        gen = input_ids[0, self.prompt_length:]
        if gen.numel() == 0:
            return False
        text = self.tokenizer.decode(gen, skip_special_tokens=False)
        return any(p.search(text) for p in self._PATTERNS)


def _has_complete_boxed_after_think(text: str) -> bool:
    """True iff there is a fully balanced `\\boxed{...}` after `</think>`."""
    if "</think>" not in text:
        return False
    post = text.split("</think>", 1)[1]
    idx = post.rfind("\\boxed{")
    if idx == -1:
        return False
    depth = 1
    i = idx + len("\\boxed{")
    while i < len(post):
        ch = post[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return True
        i += 1
    return False


class BoxedStopCriteria(StoppingCriteria):
    """Stop free-form generation once a balanced `\\boxed{...}` appears after `</think>`."""

    def __init__(self, tokenizer, prompt_length: int):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        gen = input_ids[0, self.prompt_length:]
        if gen.numel() == 0:
            return False
        text = self.tokenizer.decode(gen, skip_special_tokens=False)
        return _has_complete_boxed_after_think(text)


def strip_thinking(text: str) -> str:
    """Return only the visible answer portion of a Qwen3-Thinking output.

    Cases:
      1. `<think>...</think>ANSWER` -> return `ANSWER`.
      2. `ANSWER</think>` (budget-forced close on plain output) -> return `ANSWER`.
      3. `<think>...` (truncated mid-thinking) -> return empty string.
      4. No tags -> return text unchanged.
    """
    has_open  = "<think>"  in text
    has_close = "</think>" in text

    if has_open and has_close:
        return text.split("</think>", 1)[-1].strip()
    if has_close and not has_open:
        before, _, after = text.partition("</think>")
        return (before.strip() or after.strip())
    if has_open and not has_close:
        return ""
    return text.strip()


def clean_visible(text: str) -> str:
    """Strip thinking tags and any leftover special tokens."""
    return re.sub(r"<\|[^|]+\|>", "", strip_thinking(text)).strip()


def extract_mcq_letter(text: str) -> str:
    """Extract the MCQ letter, preferring `FINAL: X`, then `\\boxed{X}`, then any standalone A-J."""
    m = re.search(r"FINAL\s*:\s*([A-J])\b", text, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    m = re.search(r"\\boxed\{\s*([A-J])\s*\}", text, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    m = re.search(r"\b([A-J])\b", text.upper())
    return m.group(1) if m else ""


def extract_boxed(text: str) -> str:
    """Return the contents of the last simple `\\boxed{...}` (no nested-brace handling)."""
    matches = re.findall(r"\\boxed\{([^{}]+)\}", text)
    return matches[-1].strip() if matches else ""

## 6. Solver

`_generate(...)` is the only place that calls `llm.generate`. It builds the chat-templated prompt, attaches the budget-forcing processor, and optionally attaches a stopping criterion. It returns the raw decoded suffix and the number of generated tokens.

`solve_mcq(item)`:

1. Primary pass — `/no_think`, greedy (`do_sample=False`), `MAX_TOKENS_MCQ=256`, repetition penalty 1.15, `FinalAnswerStopCriteria` to stop at the first letter.
2. If the cleaned response contains no parseable letter, run a deterministic fallback pass with a strict single-letter system prompt and `FALLBACK_MAX_TOKENS=16`. This guarantees an output every time.

`solve_free(item)`:

- One pass with thinking enabled, `THINK_BUDGET_FREE=4096` enforced by `BudgetForcingProcessor`, and `BoxedStopCriteria` to stop the moment a complete `\boxed{...}` appears after `</think>`.

`solve_item(item)` dispatches by question type.

In [7]:
def _generate(
    system_prompt: str,
    user_prompt: str,
    *,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
    top_k: int,
    repetition_penalty: float,
    do_sample: bool,
    think_budget: int,
    stopping_criteria_factory=None,
) -> dict:
    """Run a single generate call with budget forcing and optional stop criterion."""
    chat = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        chat,
        return_tensors="pt",
        truncation=True,
        max_length=16384,
    ).to(llm.device)
    prompt_len = inputs["input_ids"].shape[1]

    logits_processors = [
        BudgetForcingProcessor(
            end_think_token_id=END_THINK_TOKEN_ID,
            think_budget=think_budget,
            prompt_length=prompt_len,
        )
    ]

    stopping_criteria = None
    if stopping_criteria_factory is not None:
        stopping_criteria = StoppingCriteriaList(
            [stopping_criteria_factory(tokenizer, prompt_len)]
        )

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        repetition_penalty=repetition_penalty,
        logits_processor=logits_processors,
    )
    if do_sample:
        gen_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)
    if stopping_criteria is not None:
        gen_kwargs["stopping_criteria"] = stopping_criteria

    with torch.no_grad():
        output_ids = llm.generate(**gen_kwargs)

    new_tokens = output_ids[0][prompt_len:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=False).strip()
    return {"raw": raw, "n_tokens": int(new_tokens.shape[0])}


def solve_mcq(item: dict) -> dict:
    options = item["options"]
    user = build_mcq_user(item["question"], options, no_think=MCQ_NO_THINK)

    primary = _generate(
        SYSTEM_PROMPT_MCQ,
        user,
        max_new_tokens=MAX_TOKENS_MCQ,
        temperature=TEMP_MCQ,
        top_p=TOP_P_MCQ,
        top_k=TOP_K_MCQ,
        repetition_penalty=REP_PEN_MCQ,
        do_sample=False,
        think_budget=THINK_BUDGET_MCQ,
        stopping_criteria_factory=FinalAnswerStopCriteria,
    )
    response = clean_visible(primary["raw"])
    letter = extract_mcq_letter(response)
    fallback_used = False

    if not letter:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        fb_system = (
            "You output a single uppercase letter from A to J. "
            "Nothing else. No words. No punctuation."
        )
        fb_user = (
            f"{item['question']}\n\nOptions:\n{opts_text}\n\nAnswer:/no_think"
        )
        fb = _generate(
            fb_system,
            fb_user,
            max_new_tokens=FALLBACK_MAX_TOKENS,
            temperature=0.0,
            top_p=1.0,
            top_k=0,
            repetition_penalty=1.0,
            do_sample=False,
            think_budget=THINK_BUDGET_MCQ,
            stopping_criteria_factory=None,
        )
        fb_resp = clean_visible(fb["raw"])
        letter = extract_mcq_letter(fb_resp)
        fallback_used = True

    final_response = f"FINAL: {letter}" if letter else response
    return {
        "response": final_response,
        "raw": primary["raw"],
        "meta": {
            "is_mcq": True,
            "n_tokens": primary["n_tokens"],
            "fallback_used": fallback_used,
        },
    }


def solve_free(item: dict) -> dict:
    user = build_free_user(item["question"])
    out = _generate(
        SYSTEM_PROMPT_FREE,
        user,
        max_new_tokens=MAX_TOKENS_FREE,
        temperature=TEMP_FREE,
        top_p=TOP_P_FREE,
        top_k=TOP_K_FREE,
        repetition_penalty=REP_PEN_FREE,
        do_sample=True,
        think_budget=THINK_BUDGET_FREE,
        stopping_criteria_factory=BoxedStopCriteria,
    )
    response = clean_visible(out["raw"])
    return {
        "response": response,
        "raw": out["raw"],
        "meta": {
            "is_mcq": False,
            "n_tokens": out["n_tokens"],
            "boxed": extract_boxed(response),
        },
    }


def solve_item(item: dict) -> dict:
    return solve_mcq(item) if item.get("options") else solve_free(item)

## 7. Smoke test

One MCQ + one free-form item, end-to-end. Use this to sanity-check the pipeline without paying the cost of a full run.

In [8]:
_smoke_items = [
    next(d for d in data if d.get("options")),
    next(d for d in data if not d.get("options")),
]

for _item in _smoke_items:
    _t0 = time.time()
    _solved = solve_item(_item)
    _dt = time.time() - _t0
    _kind = "MCQ" if _item.get("options") else "Free-form"
    print(f"\n── {_kind}  id={_item.get('id')}  | {_dt:.1f}s | tokens={_solved['meta']['n_tokens']} ──")
    print(f"  Gold     : {_item['answer']}")
    print(f"  Response : {_solved['response'][:200]}")
    print(f"  Meta     : {_solved['meta']}")


── MCQ  id=1  | 18.9s | tokens=256 ──
  Gold     : F
  Response : FINAL: A
  Meta     : {'is_mcq': True, 'n_tokens': 256, 'fallback_used': False}

── Free-form  id=0  | 141.5s | tokens=1989 ──
  Gold     : ['325*(1+325)']
  Response : Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let's recall what the positive even whole numbers are. The first few are 2, 4, 6, ื่, so they start at
  Meta     : {'is_mcq': False, 'n_tokens': 1989, 'boxed': ''}


## 8. Full run

One linear loop over the dataset. Set `LIMIT` in the config cell to run on a slice instead.

In [11]:
_items = data if LIMIT is None else data[:LIMIT]

responses = []
_start = time.time()

for _item in tqdm(_items, desc="Solving"):
    solved = solve_item(_item)
    responses.append({
        "item":     _item,
        "label":    "MCQ" if _item.get("options") else "Free-form",
        "response": solved["response"],
        "meta":     solved["meta"],
    })

_elapsed = time.time() - _start
print(f"Generated {len(responses)} answers in {_elapsed/60:.1f} min ({_elapsed/max(1,len(responses)):.2f} s/item)")

Solving: 100%|██████████| 5/5 [08:13<00:00, 98.74s/it] 

Generated 5 answers in 8.2 min (98.74 s/item)


## 9. Score

MCQ uses an exact-letter match. Free-form uses `Judger.auto_judge`, which handles symbolic and numeric equivalence (e.g. `325*(1+325) == 105950`).

In [12]:
_cwd = Path.cwd()
if (_cwd / "judger.py").exists():
    sys.path.insert(0, str(_cwd))
elif (_cwd.parent / "judger.py").exists():
    sys.path.insert(0, str(_cwd.parent))
else:
    raise ModuleNotFoundError("Could not locate judger.py from current working directory")

from judger import Judger
judger = Judger(strict_extract=False)

results = []
for r in tqdm(responses, desc="Scoring"):
    item     = r["item"]
    response = r["response"]
    is_mcq   = bool(item.get("options"))
    gold     = item["answer"]

    if is_mcq:
        correct = (extract_mcq_letter(response) == str(gold).strip().upper())
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
        "meta":     r.get("meta", {}),
    })

print(f"Scored {len(results)} responses")

Scoring: 100%|██████████| 5/5 [00:00<00:00, 156.01it/s]

Scored 5 responses


## 10. Summary

Per-type accuracy plus diagnostic counters that tell you whether the safeguards are firing as intended.

In [13]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def _acc(subset):
    return (sum(r["correct"] for r in subset) / len(subset) * 100.0) if subset else 0.0

print("=" * 60)
print("MODULAR PIPELINE RESULTS")
print("=" * 60)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({_acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({_acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({_acc(results):.2f}%)")
print("=" * 60)

_mcq_no_letter   = sum(1 for r in mcq_res  if not extract_mcq_letter(r["response"]))
_mcq_fallback    = sum(1 for r in mcq_res  if r.get("meta", {}).get("fallback_used"))
_free_no_box     = sum(1 for r in free_res if not r.get("meta", {}).get("boxed"))
_avg_mcq_tokens  = (sum(r.get("meta", {}).get("n_tokens", 0) for r in mcq_res)  / max(1, len(mcq_res)))
_avg_free_tokens = (sum(r.get("meta", {}).get("n_tokens", 0) for r in free_res) / max(1, len(free_res)))

print(f"  MCQ no-letter (after fallback) : {_mcq_no_letter}")
print(f"  MCQ fallback used              : {_mcq_fallback}")
print(f"  Free-form no \\boxed{{}}        : {_free_no_box}")
print(f"  Avg tokens / MCQ               : {_avg_mcq_tokens:.0f}")
print(f"  Avg tokens / Free-form         : {_avg_free_tokens:.0f}")

MODULAR PIPELINE RESULTS
  MCQ        :    0 /    2  (0.00%)
  Free-form  :    1 /    3  (33.33%)
  Overall    :    1 /    5  (20.00%)
  MCQ no-letter (after fallback) : 0
  MCQ fallback used              : 0
  Free-form no \boxed{}        : 3
  Avg tokens / MCQ               : 256
  Avg tokens / Free-form         : 2148


## 11. Save results

JSONL with `{id, is_mcq, gold, response, correct}` when `SAVE_EVAL=True`, otherwise `{id, is_mcq, response}` for private-test submission.

In [14]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {
                "id":       r["id"],
                "is_mcq":   r["is_mcq"],
                "gold":     r["gold"],
                "response": r["response"],
                "correct":  r["correct"],
            }
        else:
            record = {
                "id":       r["id"],
                "is_mcq":   r["is_mcq"],
                "response": r["response"],
            }
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 5 records to results/modular_results.jsonl
